# Ants vs Bees Binary Classification (Ultralytics YOLO)

This notebook trains a binary classifier on the Hymenoptera (Ants vs Bees) dataset.

The dataset is small enough to run on CPU/MPS while still showing end-to-end classification training.
Set the paths below to point at your local dataset folder before running the prep cell.


## 1. Install Dependencies

Run once per environment. If you already installed the repo requirements, you can skip this cell.


In [ ]:
!pip install -r ../requirements.txt

In [ ]:
from pathlib import Path
import os
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display
from sklearn.metrics import classification_report, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from ultralytics import YOLO

## 2. Dataset Setup

Download the Hymenoptera dataset (Ants vs Bees) and place it locally.

- Dataset zip: https://download.pytorch.org/tutorial/hymenoptera_data.zip

Expected raw layout after unzip:

```
ants_bees_raw/
└── hymenoptera_data/
    ├── train/
    │   ├── ants/
    │   └── bees/
    └── val/
        ├── ants/
        └── bees/
```

Set `RAW_DATA_DIR` to the folder containing `hymenoptera_data/`.

### YOLO Classification Folder Format

Ultralytics YOLO expects classification datasets as folders split by `train/`, `val/`, and `test/`, with one subfolder per class:

```
OUTPUT_DIR/
├── train/
│   ├── ants/
│   └── bees/
├── val/
│   ├── ants/
│   └── bees/
└── test/
    ├── ants/
    └── bees/
```

The prep cell below creates this structure by splitting the training set into train/val and using the original `val/` folder as test.


In [ ]:
# Optional: download + unzip the dataset automatically.
# Set DOWNLOAD = True if you want the notebook to fetch the zip for you.
DOWNLOAD = True
RAW_DATA_DIR = Path("/Users/juanterven/data/ants_bees_raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

zip_path = RAW_DATA_DIR / "hymenoptera_data.zip"
dataset_dir = RAW_DATA_DIR / "hymenoptera_data"

if DOWNLOAD and not dataset_dir.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(
        "https://download.pytorch.org/tutorial/hymenoptera_data.zip",
        zip_path,
    )
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(RAW_DATA_DIR)
    print("Done.")

In [ ]:
# Dataset preparation
RAW_DATA_DIR = Path("/Users/juanterven/data/ants_bees_raw")
OUTPUT_DIR = Path("/Users/juanterven/data/ants_bees_yolo")
SPLIT_SEED = 42
VAL_FRACTION = 0.1
MAX_SAMPLES_PER_CLASS = None  # e.g., 200 for a quick demo


def safe_link_or_copy(src: Path, dst: Path) -> None:
    """Create a symlink (or copy) from src to dst, ensuring dst's parent exists."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def build_split_dataframe(split_dir: Path, class_names: list[str]) -> pd.DataFrame:
    """Collect image paths and labels from a split directory."""
    rows = []
    for class_name in class_names:
        class_dir = split_dir / class_name
        if not class_dir.exists():
            continue
        for image_path in class_dir.rglob("*"):
            if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            rows.append({"path": image_path, "label": class_name})
    return pd.DataFrame(rows)


def sample_per_class(df: pd.DataFrame, max_samples: int | None) -> pd.DataFrame:
    """Optionally subsample each class to a maximum number of samples."""
    if max_samples is None:
        return df
    return (
        df.groupby("label", group_keys=False)
        .apply(lambda group: group.sample(min(len(group), max_samples), random_state=SPLIT_SEED))
        .reset_index(drop=True)
    )


dataset_root = RAW_DATA_DIR / "hymenoptera_data"
train_dir = dataset_root / "train"
test_dir = dataset_root / "val"

if not train_dir.exists() or not test_dir.exists():
    raise FileNotFoundError("Expected train/val folders under hymenoptera_data.")

class_names = sorted([path.name for path in train_dir.iterdir() if path.is_dir()])
train_df = build_split_dataframe(train_dir, class_names)
test_df = build_split_dataframe(test_dir, class_names)

train_df = sample_per_class(train_df, MAX_SAMPLES_PER_CLASS)
train_df, val_df = train_test_split(
    train_df, test_size=VAL_FRACTION, random_state=SPLIT_SEED, stratify=train_df["label"]
)


def write_split(split_df: pd.DataFrame, split_name: str) -> None:
    """Write a dataset split to the YOLO folder structure."""
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"{split_name} split"):
        src = Path(row["path"])
        dst = OUTPUT_DIR / split_name / row["label"] / src.name
        safe_link_or_copy(src, dst)


if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_split(train_df, "train")
write_split(val_df, "val")
write_split(test_df, "test")

print("Dataset prepared at", OUTPUT_DIR.resolve())
print("Classes:", class_names)
print("Train samples:", len(train_df))
print("Val samples:", len(val_df))
print("Test samples:", len(test_df))

## 3. Train the Classifier

If you do not have a GPU, set `device='cpu'` and reduce the batch size.

### Output Structure

Training outputs are saved under the `runs/` directory (relative to the notebook):

```
runs/
└── ants_bees_cls/
    ├── weights/
    │   ├── best.pt
    │   └── last.pt
    ├── results.csv
    └── ...
```

Use `best.pt` for evaluation and inference, or change the run name via `name=...` in the train call.


In [ ]:
device = "mps"  # "mps" (Apple Silicon), "cpu", or "cuda:0"

model = YOLO("yolo26n-cls.pt")
results = model.train(
    data=str(OUTPUT_DIR),  # path to YOLO-formatted dataset folder
    epochs=100,  # maximum training epochs
    patience=20,  # stop if no improvement for 10 epochs
    imgsz=224,  # input image size
    batch=64,  # batch size
    device=device,
    project="runs",  # base output directory
    name="ants_bees_cls"  # run name under project
)

## 4. Evaluate on the Test Split + Quick Inference

For a CLI-only workflow, you can also run:

```bash
python scripts/classify/test_ants_bees.py --model runs/ants_bees_cls/weights/best.pt --data /Users/juanterven/data/ants_bees_yolo
```


In [ ]:
metrics = model.val(
    data=str(OUTPUT_DIR),
    split="test",
    project="/Users/juanterven/dev/yolo_course/notebooks/runs",
    name="ants_bees_eval"
)
print(metrics)

# compute the precision and recall using scikit-learn
image_paths = [
    path
    for path in (OUTPUT_DIR / "test").rglob("*")
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]
class_labels = sorted({path.parent.name for path in image_paths})

y_true = []
y_pred = []
for image_path in tqdm(image_paths, desc="Predicting test images"):
    y_true.append(image_path.parent.name)
    preds = model.predict(source=str(image_path), verbose=False)
    top1 = int(preds[0].probs.top1)
    y_pred.append(preds[0].names[top1])

precision = precision_score(y_true, y_pred, labels=class_labels, average="macro")
recall = recall_score(y_true, y_pred, labels=class_labels, average="macro")

print(f"Precision (macro): {precision:.4f}")
print(f"Recall (macro): {recall:.4f}")
print(classification_report(y_true, y_pred, labels=class_labels, target_names=class_labels))

## 5. Random Test Image + Probabilities

Run this cell to sample a new test image, display it, and print class probabilities.

In [ ]:
model = YOLO("/Users/juanterven/dev/yolo_course/notebooks/runs/ants_bees_cls2/weights/best.pt")

In [ ]:
# Pick a random test image each time
random_image = random.choice(list((OUTPUT_DIR / "test").rglob("*.jpg")))
print(random_image)

# Display the image (explicit display helps some notebook renderers)
image = Image.open(random_image)
display(image)

# Run prediction
preds = model.predict(source=str(random_image), verbose=False)
probs = preds[0].probs

if probs is None:
    raise ValueError("No probabilities returned. Ensure you are using a classification model.")

# Map class indices to names
class_names = preds[0].names
scores = probs.data.cpu().numpy()

print("Prediction probabilities:")
for class_id, score in enumerate(scores):
    class_name = class_names.get(class_id, str(class_id))
    print(f"  {class_name}: {score:.4f}")